# RL ETH Engine - Institutional Industrial Monitor

## 🛑 CAPA 0: Sincronización Dura (GitHub Source of Truth)
Cada corrida arranca con un reset limpio para evitar código viejo.

In [ ]:
%%bash
set -e
REPO_URL="https://github.com/oscar0rdz/rl_eth_engine.git"
WORKDIR="/content/rl_eth_engine"
BRANCH="main"

if [ -d "$WORKDIR/.git" ]; then
  cd "$WORKDIR"
  git fetch origin
  git reset --hard "origin/${BRANCH}"
  git clean -fd
else
  rm -rf "$WORKDIR"
  git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"
  cd "$WORKDIR"
fi

echo "REPO SYNCED AT COMMIT: $(git rev-parse HEAD)"
git status --short

## ⚙️ CAPA 1: Preparación y Validación
Instalación de dependencias y ejecución de `verify_project.sh`.

In [ ]:
%%bash
set -e
cd /content/rl_eth_engine
pip install -r requirements.txt

# Verificar GPU
nvidia-smi
python3 -c "import torch; print('CUDA Available:', torch.cuda.is_available())"

# Validación industrial
export COLAB_ENV=true
bash scripts/verify_project.sh

## 🚀 CAPA 2: Lanzamiento Pipeline Gated
Entrenamiento con redirección de logs dual (consola + archivo).

In [ ]:
%%bash
set -e
cd /content/rl_eth_engine
mkdir -p logs

# Lanzar Gate 1 (o Gate 2)
python3 training/run_v4_gated_v1.py --gate 1 2>&1 | tee logs/gate1_train.log

## 📊 CAPA 3: Visualización Dinámica
TensorBoard embebido y Monitor de Resultados CSV.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/rl_eth_engine/tensorboard_logs

In [ ]:
import pandas as pd
import json
import os
from IPython.display import display, clear_output, HTML
import time

repo_path = '/content/rl_eth_engine'
csv_path = f'{repo_path}/evaluation/industrial_results.csv'
status_path = f'{repo_path}/logs/status.json'
done_flag = f'{repo_path}/logs/done.flag'
error_flag = f'{repo_path}/logs/error.flag'

def monitor_loop():
    while True:
        clear_output(wait=True)
        
        # 1. Estado Actual
        if os.path.exists(status_path):
            with open(status_path, 'r') as f:
                status = json.load(f)
                print(f"🔹 [ESTADO] Gate: {status['gate']} | Seed: {status['seed']} | Win: {status['window']} | Step: {status['step']}/{status['total_steps']}")
                print(f"🕒 Último Update: {status['last_update']}")
        
        # 2. Banderas de Finalización
        if os.path.exists(done_flag):
            print("✅ [TERMINADO] El pipeline ha finalizado con éxito.")
            break
        if os.path.exists(error_flag):
            print("❌ [ERROR] El pipeline ha fallado. Revisa los logs.")
            break
            
        # 3. Datos de Evaluación
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            print("\n📈 Resultados Recientes:")
            display(df.tail(10))
            
        time.sleep(30)

monitor_loop()